# Session Log — Accelerometer Entropy Adaptive PPG Sampling

**Project:** EMBC 2026 poster — accelerometer compressibility/entropy as a predictor of HR estimation error  
**Dataset:** IEEE Signal Processing Cup 2015 wrist PPG (22 recordings, 125 Hz)  
**Language:** Python 3.11 / Jupyter, conda env `ppg-entropy`  

---

## Session template (copy for each new session)

```
### Session YYYY-MM-DD
**Goal:** ...  
**Done:**
- 

**Key findings:**
- 

**Blockers / open questions:**
- 

**Next session:**
- 
```

---

### Session 2026-04-18
**Goal:** Project setup and Step 1 completion (data loading + sanity check)

**Done:**
- Confirmed dataset layout: `Training_data/` (12 T1 treadmill recordings) + `TestData/` (10 arm-movement recordings) + `TrueBPM/` (ground truth for test set)
- **Row-layout discrepancy caught and fixed:** training files have `sig` shape `(6, N)` with ECG at row 0, but test files have `sig` shape `(5, N)` with no ECG (rows 0–1 = PPG, 2–4 = ACC). The TestData Readme confirms ECG was withheld by the competition. Loader now branches on `sig.shape[0]` and asserts the shape matches the group.
- **Refined from two groups to three groups** after seeing raw ACC magnitudes:
  - T1 — treadmill (rec 01–12, training, 12 recs)
  - T2 — mixed arm exercise, `TEST_S*_T01.mat` (rec 13, 14, 19, 22 — shake/stretch/push-up/run/jump, 4 recs)
  - T3 — boxing, `TEST_S*_T02.mat` (rec 15, 16, 17, 18, 20, 21 — intensive forearm/upper-arm, 6 recs)
- Registered `Python (ppg-entropy)` as a Jupyter kernel and set it as the notebook default
- Added "Expected output" markdown before every code cell so the user can sanity-check each step independently
- Notebook runs end-to-end with 0 errors and 0 window/GT count mismatches

**Key findings:**
- Mean RMS ACC magnitude by group (raw, still contains 1 g gravity DC):
  - T1 treadmill: **1.476 g** (tight range 1.31–1.72)
  - T2 mixed arm: **1.068 g** (range 0.99–1.13)
  - T3 boxing:    **1.790 g** (range 1.47–2.02)
- Critical observation: **T3 magnitude ≥ T1 magnitude**, but T3 motion is irregular while T1 is periodic. This is exactly the discrimination problem entropy should solve — supports the paper thesis, not against it.
- Two-group analysis (treadmill vs T2+T3 pooled) gave means that were essentially tied (1.48 vs 1.50), which is why the three-group split tells a cleaner narrative.

**Blockers / open questions:**
- Gravity DC bias (~1 g) is still included in raw ACC magnitude. Deferred decision: whether to subtract it (high-pass or per-window mean removal) before computing RMS. Revisit after Step 4 entropy results to see if raw vs AC-only changes anything.
- Window-to-GT alignment is currently exact for all 22 recordings — no tricky off-by-one handling needed in Step 2.

**Next session:**
- Start Step 2: bandpass-filtered FFT HR estimator (stateless, per-window)
- Use the three-group color scheme (`steelblue` / `goldenrod` / `crimson`) in all MAE plots
- Report MAE per recording and per group (T1, T2, T3 separately)


### Session 2026-04-19
**Goal:** Factor shared loading code into a reusable module so Steps 2–6 can import from one place, and briefly revisit the gravity-removal question

**Done:**
- **Decided to defer signal pre-processing** (gravity removal, filtering) to a later step. Reverted an earlier commit that added AC-only RMS computation and a 5th panel to `plot_raw`. Step 1 is back to its pre-gravity state: raw ACC throughout, with a note in the title that pre-processing is deferred.
- Created **`data_loader.py`** as the single source of truth for:
  - Constants: `FS`, `WIN_SAMPLES`, `SHIFT`, `TRAIN_DIR`, `TEST_DIR`, `TRUEBPM_DIR`
  - Maps: `GROUP_COLORS`, `GROUP_LABELS`
  - Functions: `build_manifest()`, `load_recording()`, `rms_acc_windows()`, `load_all_recordings(verbose=False)`
- Paths inside the module are resolved relative to `__file__`, so the loader works regardless of the notebook's current working directory.
- Refactored `step1_data_loading.ipynb` to import from `data_loader.py` instead of defining the loader inline. Removed the now-redundant paths cell. Step 1 is now a lightweight sanity-check / visualization tour that reuses the same loader as every downstream step.
- Smoke-tested the refactored notebook end-to-end: all 22 recordings load, 0 window/GT mismatches, group means unchanged (T1=1.476, T2=1.068, T3=1.790).

**Key findings:**
- The AC-only demeaning *did* give a cleaner numeric separation of T1/T2/T3 group means, but deferred pending Step 4 decisions — we may need different pre-processing for HR estimation (Step 2) vs entropy metrics (Step 4).
- The conda env `ppg-entropy` and the Jupyter kernel `Python (ppg-entropy)` from yesterday are still registered and in use — no env changes today.

**Next session:**
- Create `step2_hr_estimator.ipynb`: stateless bandpass + FFT-peak HR estimator on the PPG-channel average, per 8 s window
- Compare estimated BPM against `bpm0` ground truth; report MAE per recording and per group

### Session 2026-04-19 (later)
**Goal:** Build Step 2 — stateless FFT-peak HR estimator with the same module+notebook split as Step 1

**Done:**
- Created `hr_estimator.py` containing the full Step 2 pipeline:
  - `design_bandpass`, `bandpass_filter` — 4th-order Butterworth 0.5–4 Hz with `filtfilt`
  - `average_channels` — mean of the two PPG rows
  - `estimate_hr_window`, `window_spectrum` — single-window FFT peak-pick (zero-padded to N=8192 → ≈ 0.9 BPM resolution)
  - `estimate_hr_trace` — end-to-end pipeline for one recording
  - `evaluate_recording`, `evaluate_all` — wrap the estimator and compare against `BPM0`
- Created `step2_hr_estimator.ipynb` (18 cells, sections 2.1–2.8): load data → filter response → raw-vs-filtered PPG → per-window FFT peak diagnostic → run estimator on all 22 recordings → per-recording MAE → MAE bar chart → estimated-vs-true traces → group summary table.
- Filter is applied end-to-end per recording once, *then* windows are sliced. Estimator is still stateless at the window level (each window's BPM is independent), but we avoid per-window filter transients. Noted this in the module docstring.
- Smoke-tested the notebook end-to-end with no errors.

**Key findings (baseline FFT-peak estimator results):**

| Group | n_windows | MAE (BPM) | Median err | p90 | Max |
|-------|-----------|-----------|------------|-----|-----|
| T1 — treadmill       | 1768 | **12.61** | 1.09 | 44.5 | 89.4 |
| T2 — mixed arm       | 511  | **14.76** | 1.68 | 51.2 | 100.9 |
| T3 — boxing          | 817  | **24.30** | 8.89 | 69.9 | 125.2 |
| ALL (pooled)         | 3096 | 16.05 | 1.59 | 53.2 | 125.2 |

- **Median ≪ MAE** in every group → heavy-tail failures. The estimator is usually right (median ≈ 1 BPM for T1/T2) but catastrophically wrong on motion-corrupted windows.
- **T3 boxing worst by far** (MAE 24 vs T1's 13), as expected — intense irregular arm motion produces artefacts squarely in the HR band, and the FFT locks on them.
- T3's median error (8.89) is ≫ T1's (1.09), meaning boxing fails *consistently*, not just sporadically.
- Per-recording MAE varies wildly within T1 (0.40 on rec 9, 29.77 on rec 10) — within-group variation is a signal worth studying in Step 3.

This is the behaviour the paper hinges on: a naive estimator works fine at rest/predictable motion and fails during unpredictable motion. Step 4's entropy metric should correlate with where these failures happen.

**Blockers / open questions:**
- No pre-processing of the ACC signal yet — still raw (gravity included). Decide before Step 4 whether the entropy metric runs on raw ACC, gravity-removed ACC, or a high-passed version.
- Baseline estimator is stateless per spec — good. But worth keeping in mind that a temporal smoothing post-process (e.g. median filter across consecutive windows) would mask per-window failures and distort the Step 3+ correlation analysis. Keep it stateless.

**Next session:**
- Step 3: pool all per-window `abs_error` values (from `evaluate_all`) against per-window `acc_rms_windows` from Step 1. Compute Pearson + Spearman correlations pooled and within each group. Scatter plot coloured by group.


---

## Research roadmap

**Shared code:**
- Loading, group maps, per-window RMS: [`data_loader.py`](data_loader.py)
- HR-estimation pipeline: [`hr_estimator.py`](hr_estimator.py)

Every step notebook imports from the relevant module rather than redefining functions inline.

| Step | Notebook / module | Status |
|------|-------------------|--------|
| Shared loader | `data_loader.py` | **done (2026-04-19)** — `build_manifest`, `load_recording`, `rms_acc_windows`, `load_all_recordings`, plus `FS`/`WIN_SAMPLES`/`SHIFT` and group colour/label maps |
| Shared HR estimator | `hr_estimator.py` | **done (2026-04-19)** — bandpass + FFT-peak pipeline; functions: `design_bandpass`, `bandpass_filter`, `average_channels`, `estimate_hr_window`, `window_spectrum`, `estimate_hr_trace`, `evaluate_recording`, `evaluate_all` |
| 1 — Data loading & sanity check | `step1_data_loading.ipynb` | **done (2026-04-18, refactored 2026-04-19)** — imports from `data_loader.py`; all 22 recordings load, 3-group split, row-layout fix, kernel registered |
| 2 — Simple HR estimator (FFT) | `step2_hr_estimator.ipynb` | **done (2026-04-19)** — stateless 4th-order Butterworth 0.5–4 Hz + FFT-peak pipeline; T1 MAE 12.6, T2 MAE 14.8, T3 MAE 24.3 BPM (median ≪ MAE → heavy-tail failures, as expected) |
| 3 — Exploratory plots (ACC mag vs error) | `step3_exploratory.ipynb` | not started |
| 4 — Entropy metrics | `step4_entropy.ipynb` | not started |
| 5 — Four-quadrant analysis | `step5_quadrant.ipynb` | not started |
| 6 — Summary comparison table | `step6_summary_table.ipynb` | not started |

**Standard prologue for every step notebook:**
```python
from data_loader import (
    FS, WIN_SAMPLES, SHIFT,
    GROUP_COLORS, GROUP_LABELS,
    load_all_recordings,
)
recordings = load_all_recordings()   # ~2-3 s
```

**Three activity groups** (refined after seeing the data):

| Group | Label | Color | Recordings | Raw mean ACC RMS (g) |
|-------|-------|-------|------------|----------------------|
| T1 | treadmill | `steelblue` | 12 (IDs 1–12, `DATA_*_TYPE*`) | 1.476 |
| T2 | mixed arm exercise | `goldenrod` | 4 (IDs 13, 14, 19, 22, `TEST_S*_T01`) | 1.068 |
| T3 | boxing | `crimson` | 6 (IDs 15, 16, 17, 18, 20, 21, `TEST_S*_T02`) | 1.790 |

**Core hypothesis:** ACC compressibility/entropy is a better predictor of HR estimation error than ACC magnitude alone.

**Key comparison cases under the three-group split:**
- T1 treadmill: high magnitude, **low** entropy → entropy predicts low error (magnitude wrongly predicts high error)
- T2 mixed arm: low magnitude, moderate entropy → mixed
- T3 boxing: high magnitude, **high** entropy → both predict high error (they agree here)

The critical discrimination is **T1 vs T3**: they have similar magnitudes but opposite entropies, and they should have very different HR-estimation error.